In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
# Supress Warnings

import warnings
warnings.filterwarnings('ignore')

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
#Import the required packages

import os
import calendar
from datetime import datetime
import math
import pandas as pd
import numpy as np

from IPython.display import display_markdown
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
# reading the dataset
telecom = pd.read_csv("data/train.csv")

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
# summary of the dataset:
telecom.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69999 entries, 0 to 69998
Data columns (total 172 columns):
 #    Column                    Non-Null Count  Dtype  
---   ------                    --------------  -----  
 0    id                        69999 non-null  int64  
 1    circle_id                 69999 non-null  int64  
 2    loc_og_t2o_mou            69297 non-null  float64
 3    std_og_t2o_mou            69297 non-null  float64
 4    loc_ic_t2o_mou            69297 non-null  float64
 5    last_date_of_month_6      69999 non-null  object 
 6    last_date_of_month_7      69600 non-null  object 
 7    last_date_of_month_8      69266 non-null  object 
 8    arpu_6                    69999 non-null  float64
 9    arpu_7                    69999 non-null  float64
 10   arpu_8                    69999 non-null  float64
 11   onnet_mou_6               67231 non-null  float64
 12   onnet_mou_7               67312 non-null  float64
 13   onnet_mou_8               66296 non-null  fl

In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
# view the top 5 rows of the data

telecom.head()

,id,circle_id,loc_og_t2o_mou,std_og_t2o_mou,loc_ic_t2o_mou,last_date_of_month_6,last_date_of_month_7,last_date_of_month_8,arpu_6,arpu_7,...,sachet_3g_7,sachet_3g_8,fb_user_6,fb_user_7,fb_user_8,aon,aug_vbc_3g,jul_vbc_3g,jun_vbc_3g,churn_probability
0,0,109,0.0,0.0,0.0,6/30/2014,7/31/2014,8/31/2014,31.277,87.009,...,0,0,NaN,NaN,NaN,1958,0.0,0.0,0.0,0
1,1,109,0.0,0.0,0.0,6/30/2014,7/31/2014,8/31/2014,0.000,122.787,...,0,0,NaN,1.0,NaN,710,0.0,0.0,0.0,0
2,2,109,0.0,0.0,0.0,6/30/2014,7/31/2014,8/31/2014,60.806,103.176,...,0,0,NaN,NaN,NaN,882,0.0,0.0,0.0,0
3,3,109,0.0,0.0,0.0,6/30/2014,7/31/2014,8/31/2014,156.362,205.260,...,0,0,NaN,NaN,NaN,982,0.0,0.0,0.0,0
4,4,109,0.0,0.0,0.0,6/30/2014,7/31/2014,8/31/2014,240.708,128.191,...,1,0,1.0,1.0,1.0,647,0.0,0.0,0.0,0


In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
#function to plot data as table
def PlotAsTable(df, figName):
    print(f"----------------------------------------------------------------------------------------------------- \n Note: If you dont see the table '{figName}' below,\n please ensure the Jupyter Notebook is marked Trusted (File --> Trusted Notebook) \n-----------------------------------------------------------------------------------------------------")
    display_markdown(f'''#### {figName} ''',  raw=True)
    display_markdown("---",  raw=True)
    display_markdown(df.to_markdown(index = False), raw=True)
    #return df.to_markdown(index = False)

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
# read the data dictonary
data_dict = pd.read_csv("data/data_dictionary.csv")
PlotAsTable(data_dict, "Data Dictionary" )

----------------------------------------------------------------------------------------------------- 
 Note: If you dont see the table 'Data Dictionary' below,
 please ensure the Jupyter Notebook is marked Trusted (File --> Trusted Notebook) 
-----------------------------------------------------------------------------------------------------


#### Data Dictionary 

---

| Acronyms   | Description                                                                       |
|:-----------|:----------------------------------------------------------------------------------|
| CIRCLE_ID  | Telecom circle area to which the customer belongs to                              |
| LOC        | Local calls  within same telecom circle                                           |
| STD        | STD calls  outside the calling circle                                             |
| IC         | Incoming calls                                                                    |
| OG         | Outgoing calls                                                                    |
| T2T        | Operator T to T ie within same operator mobile to mobile                          |
| T2M        | Operator T to other operator mobile                                               |
| T2O        | Operator T to other operator fixed line                                           |
| T2F        | Operator T to fixed lines of T                                                    |
| T2C        | Operator T to its own call center                                                 |
| ARPU       | Average revenue per user                                                          |
| MOU        | Minutes of usage  voice calls                                                     |
| AON        | Age on network  number of days the customer is using the operator T network       |
| ONNET      | All kind of calls within the same operator network                                |
| OFFNET     | All kind of calls outside the operator T network                                  |
| ROAM       | Indicates that customer is in roaming zone during the call                        |
| SPL        | Special calls                                                                     |
| ISD        | ISD calls                                                                         |
| RECH       | Recharge                                                                          |
| NUM        | Number                                                                            |
| AMT        | Amount in local currency                                                          |
| MAX        | Maximum                                                                           |
| DATA       | Mobile internet                                                                   |
| 3G         | G network                                                                         |
| AV         | Average                                                                           |
| VOL        | Mobile internet usage volume in MB                                                |
| 2G         | G network                                                                         |
| PCK        | Prepaid service schemes called  PACKS                                             |
| NIGHT      | Scheme to use during specific night hours only                                    |
| MONTHLY    | Service schemes with validity equivalent to a month                               |
| SACHET     | Service schemes with validity smaller than a month                                |
| *.6        | KPI for the month of June                                                         |
| *.7        | KPI for the month of July                                                         |
| *.8        | KPI for the month of August                                                       |
| FB_USER    | Service scheme to avail services of Facebook and similar social networking sites  |
| VBC        | Volume based cost  when no specific scheme is not purchased and paid as per usage |

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
# Convert followng columns to datetime
# last_date_of_month_6, last_date_of_month_7, last_date_of_month_8,
# date_of_last_rech_6, date_of_last_rech_7, date_of_last_rech_8
# date_of_last_rech_data_6, date_of_last_rech_data_7, date_of_last_rech_data_8

dateColumns = telecom.select_dtypes(include='object')

#function that converts required columns to datetime
def ConvertDateTimeColumns(df):
    for dCol in dateColumns.columns:
        df[dCol] = pd.to_datetime(df[dCol])

#update the datetime columns
ConvertDateTimeColumns(telecom)

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
#Rename Columns with Meaning full Names
# aug_vbc_3g jul_vbc_3g jun_vbc_3g

#function that renames required columns
def ApplyMeaningfulName(df):
    df.rename(columns={'jun_vbc_3g': 'vbc_3g_6', 'jul_vbc_3g': 'vbc_3g_7', 'aug_vbc_3g': 'vbc_3g_8'}, inplace=True)

ApplyMeaningfulName(telecom)

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
def ShowSummary(df):
    df.info(verbose=True, show_counts=True)
# summary of the updated dataset:
ShowSummary(telecom)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69999 entries, 0 to 69998
Data columns (total 172 columns):
 #    Column                    Non-Null Count  Dtype         
---   ------                    --------------  -----         
 0    id                        69999 non-null  int64         
 1    circle_id                 69999 non-null  int64         
 2    loc_og_t2o_mou            69297 non-null  float64       
 3    std_og_t2o_mou            69297 non-null  float64       
 4    loc_ic_t2o_mou            69297 non-null  float64       
 5    last_date_of_month_6      69999 non-null  datetime64[ns]
 6    last_date_of_month_7      69600 non-null  datetime64[ns]
 7    last_date_of_month_8      69266 non-null  datetime64[ns]
 8    arpu_6                    69999 non-null  float64       
 9    arpu_7                    69999 non-null  float64       
 10   arpu_8                    69999 non-null  float64       
 11   onnet_mou_6               67231 non-null  float64       
 12   on

In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
telecom.describe()

,id,circle_id,loc_og_t2o_mou,std_og_t2o_mou,loc_ic_t2o_mou,last_date_of_month_6,last_date_of_month_7,last_date_of_month_8,arpu_6,arpu_7,...,sachet_3g_7,sachet_3g_8,fb_user_6,fb_user_7,fb_user_8,aon,vbc_3g_8,vbc_3g_7,vbc_3g_6,churn_probability
count,69999.000000,69999.0,69297.0,69297.0,69297.0,69999,69600,69266,69999.000000,69999.000000,...,69999.000000,69999.000000,17568.000000,17865.000000,18417.000000,69999.000000,69999.000000,69999.000000,69999.00000,69999.000000
mean,34999.000000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00.000000256,2014-08-30 23:59:59.999999744,283.134365,278.185912,...,0.081444,0.085487,0.916325,0.909544,0.890319,1220.639709,68.108597,65.935830,60.07674,0.101887
min,0.000000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-31 00:00:00,-2258.709000,-1289.715000,...,0.000000,0.000000,0.000000,0.000000,0.000000,180.000000,0.000000,0.000000,0.00000,0.000000
25%,17499.500000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-31 00:00:00,93.581000,86.714000,...,0.000000,0.000000,1.000000,1.000000,1.000000,468.000000,0.000000,0.000000,0.00000,0.000000
50%,34999.000000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-31 00:00:00,197.484000,191.588000,...,0.000000,0.000000,1.000000,1.000000,1.000000,868.000000,0.000000,0.000000,0.00000,0.000000
75%,52498.500000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-31 00:00:00,370.791000,365.369500,...,0.000000,0.000000,1.000000,1.000000,1.000000,1813.000000,0.000000,0.000000,0.00000,0.000000
max,69998.000000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-31 00:00:00,27731.088000,35145.834000,...,33.000000,41.000000,1.000000,1.000000,1.000000,4337.000000,12916.220000,9165.600000,11166.21000,1.000000
std,20207.115084,0.0,0.0,0.0,0.0,NaN,NaN,NaN,334.213918,344.366927,...,0.634547,0.680035,0.276907,0.286842,0.312501,952.426321,269.328659,267.899034,257.22681,0.302502


In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
colDesrciption = []
data_dict['Acronyms'] = data_dict['Acronyms'].str.replace(" ","")
for colName in telecom.columns:
    colDescr = ""
    acronymDescription = data_dict.loc[data_dict['Acronyms'] == colName.upper()].head(1)
    if (not acronymDescription.empty):
        colDescr = f"{acronymDescription.iloc[0]['Description']}"
    else:
        for acronyms in colName.split("_"):
            acronymDescription = data_dict.loc[data_dict['Acronyms'] == acronyms.upper()].head(1)
            if (not acronymDescription.empty):
                colDescr = f" {colDescr} {acronyms}: {acronymDescription.iloc[0]['Description']},"
            elif (acronyms in ["6" , "7", "8"]):
                colDescr = f" {colDescr} {acronyms}: {calendar.month_name[int(acronyms)]},"
            elif (acronyms == "fb"):
                colDescr = f" {colDescr} fb_user: Service scheme to avail services of Facebook and similar social networking sites"
    colDesrciption.append(colDescr)

columnDescription = pd.DataFrame({'ColumnName': telecom.columns, 'Description':colDesrciption})
PlotAsTable(columnDescription, "Column Description" )

----------------------------------------------------------------------------------------------------- 
 Note: If you dont see the table 'Column Description' below,
 please ensure the Jupyter Notebook is marked Trusted (File --> Trusted Notebook) 
-----------------------------------------------------------------------------------------------------


#### Column Description 

---

| ColumnName               | Description                                                                                                                                                                     |
|:-------------------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| id                       |                                                                                                                                                                                 |
| circle_id                | Telecom circle area to which the customer belongs to                                                                                                                            |
| loc_og_t2o_mou           | loc: Local calls  within same telecom circle, og: Outgoing calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls,                             |
| std_og_t2o_mou           | std: STD calls  outside the calling circle, og: Outgoing calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls,                               |
| loc_ic_t2o_mou           | loc: Local calls  within same telecom circle, ic: Incoming calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls,                             |
| last_date_of_month_6     | 6: June,                                                                                                                                                                        |
| last_date_of_month_7     | 7: July,                                                                                                                                                                        |
| last_date_of_month_8     | 8: August,                                                                                                                                                                      |
| arpu_6                   | arpu: Average revenue per user, 6: June,                                                                                                                                        |
| arpu_7                   | arpu: Average revenue per user, 7: July,                                                                                                                                        |
| arpu_8                   | arpu: Average revenue per user, 8: August,                                                                                                                                      |
| onnet_mou_6              | onnet: All kind of calls within the same operator network, mou: Minutes of usage  voice calls, 6: June,                                                                         |
| onnet_mou_7              | onnet: All kind of calls within the same operator network, mou: Minutes of usage  voice calls, 7: July,                                                                         |
| onnet_mou_8              | onnet: All kind of calls within the same operator network, mou: Minutes of usage  voice calls, 8: August,                                                                       |
| offnet_mou_6             | offnet: All kind of calls outside the operator T network, mou: Minutes of usage  voice calls, 6: June,                                                                          |
| offnet_mou_7             | offnet: All kind of calls outside the operator T network, mou: Minutes of usage  voice calls, 7: July,                                                                          |
| offnet_mou_8             | offnet: All kind of calls outside the operator T network, mou: Minutes of usage  voice calls, 8: August,                                                                        |
| roam_ic_mou_6            | roam: Indicates that customer is in roaming zone during the call, ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                              |
| roam_ic_mou_7            | roam: Indicates that customer is in roaming zone during the call, ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                              |
| roam_ic_mou_8            | roam: Indicates that customer is in roaming zone during the call, ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                            |
| roam_og_mou_6            | roam: Indicates that customer is in roaming zone during the call, og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                              |
| roam_og_mou_7            | roam: Indicates that customer is in roaming zone during the call, og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                              |
| roam_og_mou_8            | roam: Indicates that customer is in roaming zone during the call, og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                            |
| loc_og_t2t_mou_6         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 6: June,   |
| loc_og_t2t_mou_7         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 7: July,   |
| loc_og_t2t_mou_8         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 8: August, |
| loc_og_t2m_mou_6         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 6: June,                        |
| loc_og_t2m_mou_7         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 7: July,                        |
| loc_og_t2m_mou_8         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 8: August,                      |
| loc_og_t2f_mou_6         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 6: June,                             |
| loc_og_t2f_mou_7         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 7: July,                             |
| loc_og_t2f_mou_8         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 8: August,                           |
| loc_og_t2c_mou_6         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 6: June,                          |
| loc_og_t2c_mou_7         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 7: July,                          |
| loc_og_t2c_mou_8         | loc: Local calls  within same telecom circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 8: August,                        |
| loc_og_mou_6             | loc: Local calls  within same telecom circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                                                  |
| loc_og_mou_7             | loc: Local calls  within same telecom circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                                                  |
| loc_og_mou_8             | loc: Local calls  within same telecom circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                                                |
| std_og_t2t_mou_6         | std: STD calls  outside the calling circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 6: June,     |
| std_og_t2t_mou_7         | std: STD calls  outside the calling circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 7: July,     |
| std_og_t2t_mou_8         | std: STD calls  outside the calling circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 8: August,   |
| std_og_t2m_mou_6         | std: STD calls  outside the calling circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 6: June,                          |
| std_og_t2m_mou_7         | std: STD calls  outside the calling circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 7: July,                          |
| std_og_t2m_mou_8         | std: STD calls  outside the calling circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 8: August,                        |
| std_og_t2f_mou_6         | std: STD calls  outside the calling circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 6: June,                               |
| std_og_t2f_mou_7         | std: STD calls  outside the calling circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 7: July,                               |
| std_og_t2f_mou_8         | std: STD calls  outside the calling circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 8: August,                             |
| std_og_t2c_mou_6         | std: STD calls  outside the calling circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 6: June,                            |
| std_og_t2c_mou_7         | std: STD calls  outside the calling circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 7: July,                            |
| std_og_t2c_mou_8         | std: STD calls  outside the calling circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 8: August,                          |
| std_og_mou_6             | std: STD calls  outside the calling circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                                                    |
| std_og_mou_7             | std: STD calls  outside the calling circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                                                    |
| std_og_mou_8             | std: STD calls  outside the calling circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                                                  |
| isd_og_mou_6             | isd: ISD calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                                                                                |
| isd_og_mou_7             | isd: ISD calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                                                                                |
| isd_og_mou_8             | isd: ISD calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                                                                              |
| spl_og_mou_6             | spl: Special calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                                                                            |
| spl_og_mou_7             | spl: Special calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                                                                            |
| spl_og_mou_8             | spl: Special calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                                                                          |
| og_others_6              | og: Outgoing calls, 6: June,                                                                                                                                                    |
| og_others_7              | og: Outgoing calls, 7: July,                                                                                                                                                    |
| og_others_8              | og: Outgoing calls, 8: August,                                                                                                                                                  |
| total_og_mou_6           | og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                                                                                                |
| total_og_mou_7           | og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                                                                                                |
| total_og_mou_8           | og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                                                                                              |
| loc_ic_t2t_mou_6         | loc: Local calls  within same telecom circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 6: June,   |
| loc_ic_t2t_mou_7         | loc: Local calls  within same telecom circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 7: July,   |
| loc_ic_t2t_mou_8         | loc: Local calls  within same telecom circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 8: August, |
| loc_ic_t2m_mou_6         | loc: Local calls  within same telecom circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 6: June,                        |
| loc_ic_t2m_mou_7         | loc: Local calls  within same telecom circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 7: July,                        |
| loc_ic_t2m_mou_8         | loc: Local calls  within same telecom circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 8: August,                      |
| loc_ic_t2f_mou_6         | loc: Local calls  within same telecom circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 6: June,                             |
| loc_ic_t2f_mou_7         | loc: Local calls  within same telecom circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 7: July,                             |
| loc_ic_t2f_mou_8         | loc: Local calls  within same telecom circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 8: August,                           |
| loc_ic_mou_6             | loc: Local calls  within same telecom circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                                                  |
| loc_ic_mou_7             | loc: Local calls  within same telecom circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                                                  |
| loc_ic_mou_8             | loc: Local calls  within same telecom circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                                                |
| std_ic_t2t_mou_6         | std: STD calls  outside the calling circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 6: June,     |
| std_ic_t2t_mou_7         | std: STD calls  outside the calling circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 7: July,     |
| std_ic_t2t_mou_8         | std: STD calls  outside the calling circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 8: August,   |
| std_ic_t2m_mou_6         | std: STD calls  outside the calling circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 6: June,                          |
| std_ic_t2m_mou_7         | std: STD calls  outside the calling circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 7: July,                          |
| std_ic_t2m_mou_8         | std: STD calls  outside the calling circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 8: August,                        |
| std_ic_t2f_mou_6         | std: STD calls  outside the calling circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 6: June,                               |
| std_ic_t2f_mou_7         | std: STD calls  outside the calling circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 7: July,                               |
| std_ic_t2f_mou_8         | std: STD calls  outside the calling circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 8: August,                             |
| std_ic_t2o_mou_6         | std: STD calls  outside the calling circle, ic: Incoming calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls, 6: June,                      |
| std_ic_t2o_mou_7         | std: STD calls  outside the calling circle, ic: Incoming calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls, 7: July,                      |
| std_ic_t2o_mou_8         | std: STD calls  outside the calling circle, ic: Incoming calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls, 8: August,                    |
| std_ic_mou_6             | std: STD calls  outside the calling circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                                                    |
| std_ic_mou_7             | std: STD calls  outside the calling circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                                                    |
| std_ic_mou_8             | std: STD calls  outside the calling circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                                                  |
| total_ic_mou_6           | ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                                                                                                |
| total_ic_mou_7           | ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                                                                                                |
| total_ic_mou_8           | ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                                                                                              |
| spl_ic_mou_6             | spl: Special calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                                                                            |
| spl_ic_mou_7             | spl: Special calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                                                                            |
| spl_ic_mou_8             | spl: Special calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                                                                          |
| isd_ic_mou_6             | isd: ISD calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                                                                                |
| isd_ic_mou_7             | isd: ISD calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                                                                                |
| isd_ic_mou_8             | isd: ISD calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                                                                              |
| ic_others_6              | ic: Incoming calls, 6: June,                                                                                                                                                    |
| ic_others_7              | ic: Incoming calls, 7: July,                                                                                                                                                    |
| ic_others_8              | ic: Incoming calls, 8: August,                                                                                                                                                  |
| total_rech_num_6         | rech: Recharge, num: Number, 6: June,                                                                                                                                           |
| total_rech_num_7         | rech: Recharge, num: Number, 7: July,                                                                                                                                           |
| total_rech_num_8         | rech: Recharge, num: Number, 8: August,                                                                                                                                         |
| total_rech_amt_6         | rech: Recharge, amt: Amount in local currency, 6: June,                                                                                                                         |
| total_rech_amt_7         | rech: Recharge, amt: Amount in local currency, 7: July,                                                                                                                         |
| total_rech_amt_8         | rech: Recharge, amt: Amount in local currency, 8: August,                                                                                                                       |
| max_rech_amt_6           | max: Maximum, rech: Recharge, amt: Amount in local currency, 6: June,                                                                                                           |
| max_rech_amt_7           | max: Maximum, rech: Recharge, amt: Amount in local currency, 7: July,                                                                                                           |
| max_rech_amt_8           | max: Maximum, rech: Recharge, amt: Amount in local currency, 8: August,                                                                                                         |
| date_of_last_rech_6      | rech: Recharge, 6: June,                                                                                                                                                        |
| date_of_last_rech_7      | rech: Recharge, 7: July,                                                                                                                                                        |
| date_of_last_rech_8      | rech: Recharge, 8: August,                                                                                                                                                      |
| last_day_rch_amt_6       | amt: Amount in local currency, 6: June,                                                                                                                                         |
| last_day_rch_amt_7       | amt: Amount in local currency, 7: July,                                                                                                                                         |
| last_day_rch_amt_8       | amt: Amount in local currency, 8: August,                                                                                                                                       |
| date_of_last_rech_data_6 | rech: Recharge, data: Mobile internet, 6: June,                                                                                                                                 |
| date_of_last_rech_data_7 | rech: Recharge, data: Mobile internet, 7: July,                                                                                                                                 |
| date_of_last_rech_data_8 | rech: Recharge, data: Mobile internet, 8: August,                                                                                                                               |
| total_rech_data_6        | rech: Recharge, data: Mobile internet, 6: June,                                                                                                                                 |
| total_rech_data_7        | rech: Recharge, data: Mobile internet, 7: July,                                                                                                                                 |
| total_rech_data_8        | rech: Recharge, data: Mobile internet, 8: August,                                                                                                                               |
| max_rech_data_6          | max: Maximum, rech: Recharge, data: Mobile internet, 6: June,                                                                                                                   |
| max_rech_data_7          | max: Maximum, rech: Recharge, data: Mobile internet, 7: July,                                                                                                                   |
| max_rech_data_8          | max: Maximum, rech: Recharge, data: Mobile internet, 8: August,                                                                                                                 |
| count_rech_2g_6          | rech: Recharge, 2g: G network, 6: June,                                                                                                                                         |
| count_rech_2g_7          | rech: Recharge, 2g: G network, 7: July,                                                                                                                                         |
| count_rech_2g_8          | rech: Recharge, 2g: G network, 8: August,                                                                                                                                       |
| count_rech_3g_6          | rech: Recharge, 3g: G network, 6: June,                                                                                                                                         |
| count_rech_3g_7          | rech: Recharge, 3g: G network, 7: July,                                                                                                                                         |
| count_rech_3g_8          | rech: Recharge, 3g: G network, 8: August,                                                                                                                                       |
| av_rech_amt_data_6       | av: Average, rech: Recharge, amt: Amount in local currency, data: Mobile internet, 6: June,                                                                                     |
| av_rech_amt_data_7       | av: Average, rech: Recharge, amt: Amount in local currency, data: Mobile internet, 7: July,                                                                                     |
| av_rech_amt_data_8       | av: Average, rech: Recharge, amt: Amount in local currency, data: Mobile internet, 8: August,                                                                                   |
| vol_2g_mb_6              | vol: Mobile internet usage volume in MB, 2g: G network, 6: June,                                                                                                                |
| vol_2g_mb_7              | vol: Mobile internet usage volume in MB, 2g: G network, 7: July,                                                                                                                |
| vol_2g_mb_8              | vol: Mobile internet usage volume in MB, 2g: G network, 8: August,                                                                                                              |
| vol_3g_mb_6              | vol: Mobile internet usage volume in MB, 3g: G network, 6: June,                                                                                                                |
| vol_3g_mb_7              | vol: Mobile internet usage volume in MB, 3g: G network, 7: July,                                                                                                                |
| vol_3g_mb_8              | vol: Mobile internet usage volume in MB, 3g: G network, 8: August,                                                                                                              |
| arpu_3g_6                | arpu: Average revenue per user, 3g: G network, 6: June,                                                                                                                         |
| arpu_3g_7                | arpu: Average revenue per user, 3g: G network, 7: July,                                                                                                                         |
| arpu_3g_8                | arpu: Average revenue per user, 3g: G network, 8: August,                                                                                                                       |
| arpu_2g_6                | arpu: Average revenue per user, 2g: G network, 6: June,                                                                                                                         |
| arpu_2g_7                | arpu: Average revenue per user, 2g: G network, 7: July,                                                                                                                         |
| arpu_2g_8                | arpu: Average revenue per user, 2g: G network, 8: August,                                                                                                                       |
| night_pck_user_6         | night: Scheme to use during specific night hours only, pck: Prepaid service schemes called  PACKS, 6: June,                                                                     |
| night_pck_user_7         | night: Scheme to use during specific night hours only, pck: Prepaid service schemes called  PACKS, 7: July,                                                                     |
| night_pck_user_8         | night: Scheme to use during specific night hours only, pck: Prepaid service schemes called  PACKS, 8: August,                                                                   |
| monthly_2g_6             | monthly: Service schemes with validity equivalent to a month, 2g: G network, 6: June,                                                                                           |
| monthly_2g_7             | monthly: Service schemes with validity equivalent to a month, 2g: G network, 7: July,                                                                                           |
| monthly_2g_8             | monthly: Service schemes with validity equivalent to a month, 2g: G network, 8: August,                                                                                         |
| sachet_2g_6              | sachet: Service schemes with validity smaller than a month, 2g: G network, 6: June,                                                                                             |
| sachet_2g_7              | sachet: Service schemes with validity smaller than a month, 2g: G network, 7: July,                                                                                             |
| sachet_2g_8              | sachet: Service schemes with validity smaller than a month, 2g: G network, 8: August,                                                                                           |
| monthly_3g_6             | monthly: Service schemes with validity equivalent to a month, 3g: G network, 6: June,                                                                                           |
| monthly_3g_7             | monthly: Service schemes with validity equivalent to a month, 3g: G network, 7: July,                                                                                           |
| monthly_3g_8             | monthly: Service schemes with validity equivalent to a month, 3g: G network, 8: August,                                                                                         |
| sachet_3g_6              | sachet: Service schemes with validity smaller than a month, 3g: G network, 6: June,                                                                                             |
| sachet_3g_7              | sachet: Service schemes with validity smaller than a month, 3g: G network, 7: July,                                                                                             |
| sachet_3g_8              | sachet: Service schemes with validity smaller than a month, 3g: G network, 8: August,                                                                                           |
| fb_user_6                | fb_user: Service scheme to avail services of Facebook and similar social networking sites 6: June,                                                                              |
| fb_user_7                | fb_user: Service scheme to avail services of Facebook and similar social networking sites 7: July,                                                                              |
| fb_user_8                | fb_user: Service scheme to avail services of Facebook and similar social networking sites 8: August,                                                                            |
| aon                      | Age on network  number of days the customer is using the operator T network                                                                                                     |
| vbc_3g_8                 | vbc: Volume based cost  when no specific scheme is not purchased and paid as per usage, 3g: G network, 8: August,                                                               |
| vbc_3g_7                 | vbc: Volume based cost  when no specific scheme is not purchased and paid as per usage, 3g: G network, 7: July,                                                                 |
| vbc_3g_6                 | vbc: Volume based cost  when no specific scheme is not purchased and paid as per usage, 3g: G network, 6: June,                                                                 |
| churn_probability        |                                                                                                                                                                                 |

In [13]:
# --- [CELL 12]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
# see the top 5 rows of the data
telecom.head()

,id,circle_id,loc_og_t2o_mou,std_og_t2o_mou,loc_ic_t2o_mou,last_date_of_month_6,last_date_of_month_7,last_date_of_month_8,arpu_6,arpu_7,...,sachet_3g_7,sachet_3g_8,fb_user_6,fb_user_7,fb_user_8,aon,vbc_3g_8,vbc_3g_7,vbc_3g_6,churn_probability
0,0,109,0.0,0.0,0.0,2014-06-30,2014-07-31,2014-08-31,31.277,87.009,...,0,0,NaN,NaN,NaN,1958,0.0,0.0,0.0,0
1,1,109,0.0,0.0,0.0,2014-06-30,2014-07-31,2014-08-31,0.000,122.787,...,0,0,NaN,1.0,NaN,710,0.0,0.0,0.0,0
2,2,109,0.0,0.0,0.0,2014-06-30,2014-07-31,2014-08-31,60.806,103.176,...,0,0,NaN,NaN,NaN,882,0.0,0.0,0.0,0
3,3,109,0.0,0.0,0.0,2014-06-30,2014-07-31,2014-08-31,156.362,205.260,...,0,0,NaN,NaN,NaN,982,0.0,0.0,0.0,0
4,4,109,0.0,0.0,0.0,2014-06-30,2014-07-31,2014-08-31,240.708,128.191,...,1,0,1.0,1.0,1.0,647,0.0,0.0,0.0,0


In [14]:
# --- [CELL 13]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
def ReturnColumnsMissingPercentage(df):
    missing = df.isnull().sum() * 100 / len(df)
    missing.name = "MissingPercentage"
    missing = missing.to_frame().reset_index()
    return missing

percent_missing = ReturnColumnsMissingPercentage(telecom)
percent_missing["Description"] = columnDescription["Description"]
PlotAsTable(percent_missing.sort_values(by=['MissingPercentage'], ascending = False), "Missing Percentage")

----------------------------------------------------------------------------------------------------- 
 Note: If you dont see the table 'Missing Percentage' below,
 please ensure the Jupyter Notebook is marked Trusted (File --> Trusted Notebook) 
-----------------------------------------------------------------------------------------------------


#### Missing Percentage 

---

| index                    |   MissingPercentage | Description                                                                                                                                                                     |
|:-------------------------|--------------------:|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| arpu_3g_6                |           74.9025   | arpu: Average revenue per user, 3g: G network, 6: June,                                                                                                                         |
| count_rech_2g_6          |           74.9025   | rech: Recharge, 2g: G network, 6: June,                                                                                                                                         |
| night_pck_user_6         |           74.9025   | night: Scheme to use during specific night hours only, pck: Prepaid service schemes called  PACKS, 6: June,                                                                     |
| arpu_2g_6                |           74.9025   | arpu: Average revenue per user, 2g: G network, 6: June,                                                                                                                         |
| date_of_last_rech_data_6 |           74.9025   | rech: Recharge, data: Mobile internet, 6: June,                                                                                                                                 |
| total_rech_data_6        |           74.9025   | rech: Recharge, data: Mobile internet, 6: June,                                                                                                                                 |
| av_rech_amt_data_6       |           74.9025   | av: Average, rech: Recharge, amt: Amount in local currency, data: Mobile internet, 6: June,                                                                                     |
| max_rech_data_6          |           74.9025   | max: Maximum, rech: Recharge, data: Mobile internet, 6: June,                                                                                                                   |
| count_rech_3g_6          |           74.9025   | rech: Recharge, 3g: G network, 6: June,                                                                                                                                         |
| fb_user_6                |           74.9025   | fb_user: Service scheme to avail services of Facebook and similar social networking sites 6: June,                                                                              |
| night_pck_user_7         |           74.4782   | night: Scheme to use during specific night hours only, pck: Prepaid service schemes called  PACKS, 7: July,                                                                     |
| date_of_last_rech_data_7 |           74.4782   | rech: Recharge, data: Mobile internet, 7: July,                                                                                                                                 |
| total_rech_data_7        |           74.4782   | rech: Recharge, data: Mobile internet, 7: July,                                                                                                                                 |
| max_rech_data_7          |           74.4782   | max: Maximum, rech: Recharge, data: Mobile internet, 7: July,                                                                                                                   |
| fb_user_7                |           74.4782   | fb_user: Service scheme to avail services of Facebook and similar social networking sites 7: July,                                                                              |
| count_rech_2g_7          |           74.4782   | rech: Recharge, 2g: G network, 7: July,                                                                                                                                         |
| count_rech_3g_7          |           74.4782   | rech: Recharge, 3g: G network, 7: July,                                                                                                                                         |
| arpu_3g_7                |           74.4782   | arpu: Average revenue per user, 3g: G network, 7: July,                                                                                                                         |
| av_rech_amt_data_7       |           74.4782   | av: Average, rech: Recharge, amt: Amount in local currency, data: Mobile internet, 7: July,                                                                                     |
| arpu_2g_7                |           74.4782   | arpu: Average revenue per user, 2g: G network, 7: July,                                                                                                                         |
| count_rech_2g_8          |           73.6896   | rech: Recharge, 2g: G network, 8: August,                                                                                                                                       |
| av_rech_amt_data_8       |           73.6896   | av: Average, rech: Recharge, amt: Amount in local currency, data: Mobile internet, 8: August,                                                                                   |
| night_pck_user_8         |           73.6896   | night: Scheme to use during specific night hours only, pck: Prepaid service schemes called  PACKS, 8: August,                                                                   |
| max_rech_data_8          |           73.6896   | max: Maximum, rech: Recharge, data: Mobile internet, 8: August,                                                                                                                 |
| total_rech_data_8        |           73.6896   | rech: Recharge, data: Mobile internet, 8: August,                                                                                                                               |
| arpu_2g_8                |           73.6896   | arpu: Average revenue per user, 2g: G network, 8: August,                                                                                                                       |
| arpu_3g_8                |           73.6896   | arpu: Average revenue per user, 3g: G network, 8: August,                                                                                                                       |
| date_of_last_rech_data_8 |           73.6896   | rech: Recharge, data: Mobile internet, 8: August,                                                                                                                               |
| fb_user_8                |           73.6896   | fb_user: Service scheme to avail services of Facebook and similar social networking sites 8: August,                                                                            |
| count_rech_3g_8          |           73.6896   | rech: Recharge, 3g: G network, 8: August,                                                                                                                                       |
| isd_og_mou_8             |            5.29008  | isd: ISD calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                                                                              |
| std_ic_t2o_mou_8         |            5.29008  | std: STD calls  outside the calling circle, ic: Incoming calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls, 8: August,                    |
| std_og_t2c_mou_8         |            5.29008  | std: STD calls  outside the calling circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 8: August,                          |
| std_ic_t2f_mou_8         |            5.29008  | std: STD calls  outside the calling circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 8: August,                             |
| std_og_mou_8             |            5.29008  | std: STD calls  outside the calling circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                                                  |
| std_ic_t2m_mou_8         |            5.29008  | std: STD calls  outside the calling circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 8: August,                        |
| std_ic_mou_8             |            5.29008  | std: STD calls  outside the calling circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                                                  |
| std_ic_t2t_mou_8         |            5.29008  | std: STD calls  outside the calling circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 8: August,   |
| og_others_8              |            5.29008  | og: Outgoing calls, 8: August,                                                                                                                                                  |
| spl_og_mou_8             |            5.29008  | spl: Special calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                                                                          |
| loc_ic_t2m_mou_8         |            5.29008  | loc: Local calls  within same telecom circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 8: August,                      |
| loc_ic_mou_8             |            5.29008  | loc: Local calls  within same telecom circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                                                |
| loc_ic_t2f_mou_8         |            5.29008  | loc: Local calls  within same telecom circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 8: August,                           |
| std_og_t2f_mou_8         |            5.29008  | std: STD calls  outside the calling circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 8: August,                             |
| loc_og_t2c_mou_8         |            5.29008  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 8: August,                        |
| ic_others_8              |            5.29008  | ic: Incoming calls, 8: August,                                                                                                                                                  |
| loc_og_mou_8             |            5.29008  | loc: Local calls  within same telecom circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                                                |
| onnet_mou_8              |            5.29008  | onnet: All kind of calls within the same operator network, mou: Minutes of usage  voice calls, 8: August,                                                                       |
| offnet_mou_8             |            5.29008  | offnet: All kind of calls outside the operator T network, mou: Minutes of usage  voice calls, 8: August,                                                                        |
| roam_ic_mou_8            |            5.29008  | roam: Indicates that customer is in roaming zone during the call, ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                            |
| roam_og_mou_8            |            5.29008  | roam: Indicates that customer is in roaming zone during the call, og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                            |
| loc_og_t2t_mou_8         |            5.29008  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 8: August, |
| loc_og_t2m_mou_8         |            5.29008  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 8: August,                      |
| loc_og_t2f_mou_8         |            5.29008  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 8: August,                           |
| std_og_t2m_mou_8         |            5.29008  | std: STD calls  outside the calling circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 8: August,                        |
| loc_ic_t2t_mou_8         |            5.29008  | loc: Local calls  within same telecom circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 8: August, |
| isd_ic_mou_8             |            5.29008  | isd: ISD calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                                                                              |
| std_og_t2t_mou_8         |            5.29008  | std: STD calls  outside the calling circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 8: August,   |
| spl_ic_mou_8             |            5.29008  | spl: Special calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                                                                          |
| std_ic_t2m_mou_6         |            3.95434  | std: STD calls  outside the calling circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 6: June,                          |
| std_ic_t2t_mou_6         |            3.95434  | std: STD calls  outside the calling circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 6: June,     |
| loc_ic_t2m_mou_6         |            3.95434  | loc: Local calls  within same telecom circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 6: June,                        |
| ic_others_6              |            3.95434  | ic: Incoming calls, 6: June,                                                                                                                                                    |
| loc_ic_mou_6             |            3.95434  | loc: Local calls  within same telecom circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                                                  |
| std_ic_t2f_mou_6         |            3.95434  | std: STD calls  outside the calling circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 6: June,                               |
| isd_ic_mou_6             |            3.95434  | isd: ISD calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                                                                                |
| std_ic_mou_6             |            3.95434  | std: STD calls  outside the calling circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                                                    |
| spl_ic_mou_6             |            3.95434  | spl: Special calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                                                                            |
| std_ic_t2o_mou_6         |            3.95434  | std: STD calls  outside the calling circle, ic: Incoming calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls, 6: June,                      |
| loc_ic_t2f_mou_6         |            3.95434  | loc: Local calls  within same telecom circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 6: June,                             |
| isd_og_mou_6             |            3.95434  | isd: ISD calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                                                                                |
| std_og_t2m_mou_6         |            3.95434  | std: STD calls  outside the calling circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 6: June,                          |
| std_og_t2f_mou_6         |            3.95434  | std: STD calls  outside the calling circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 6: June,                               |
| loc_og_mou_6             |            3.95434  | loc: Local calls  within same telecom circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                                                  |
| loc_og_t2c_mou_6         |            3.95434  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 6: June,                          |
| std_og_t2c_mou_6         |            3.95434  | std: STD calls  outside the calling circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 6: June,                            |
| loc_og_t2f_mou_6         |            3.95434  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 6: June,                             |
| loc_og_t2m_mou_6         |            3.95434  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 6: June,                        |
| std_og_mou_6             |            3.95434  | std: STD calls  outside the calling circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                                                    |
| loc_og_t2t_mou_6         |            3.95434  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 6: June,   |
| std_og_t2t_mou_6         |            3.95434  | std: STD calls  outside the calling circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 6: June,     |
| loc_ic_t2t_mou_6         |            3.95434  | loc: Local calls  within same telecom circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 6: June,   |
| spl_og_mou_6             |            3.95434  | spl: Special calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                                                                            |
| onnet_mou_6              |            3.95434  | onnet: All kind of calls within the same operator network, mou: Minutes of usage  voice calls, 6: June,                                                                         |
| roam_ic_mou_6            |            3.95434  | roam: Indicates that customer is in roaming zone during the call, ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                              |
| og_others_6              |            3.95434  | og: Outgoing calls, 6: June,                                                                                                                                                    |
| roam_og_mou_6            |            3.95434  | roam: Indicates that customer is in roaming zone during the call, og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                              |
| offnet_mou_6             |            3.95434  | offnet: All kind of calls outside the operator T network, mou: Minutes of usage  voice calls, 6: June,                                                                          |
| roam_og_mou_7            |            3.83863  | roam: Indicates that customer is in roaming zone during the call, og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                              |
| ic_others_7              |            3.83863  | ic: Incoming calls, 7: July,                                                                                                                                                    |
| loc_og_mou_7             |            3.83863  | loc: Local calls  within same telecom circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                                                  |
| onnet_mou_7              |            3.83863  | onnet: All kind of calls within the same operator network, mou: Minutes of usage  voice calls, 7: July,                                                                         |
| loc_ic_t2t_mou_7         |            3.83863  | loc: Local calls  within same telecom circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 7: July,   |
| loc_og_t2f_mou_7         |            3.83863  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 7: July,                             |
| loc_og_t2c_mou_7         |            3.83863  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 7: July,                          |
| offnet_mou_7             |            3.83863  | offnet: All kind of calls outside the operator T network, mou: Minutes of usage  voice calls, 7: July,                                                                          |
| loc_og_t2m_mou_7         |            3.83863  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 7: July,                        |
| roam_ic_mou_7            |            3.83863  | roam: Indicates that customer is in roaming zone during the call, ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                              |
| std_og_t2t_mou_7         |            3.83863  | std: STD calls  outside the calling circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 7: July,     |
| loc_og_t2t_mou_7         |            3.83863  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 7: July,   |
| loc_ic_t2m_mou_7         |            3.83863  | loc: Local calls  within same telecom circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 7: July,                        |
| isd_ic_mou_7             |            3.83863  | isd: ISD calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                                                                                |
| loc_ic_t2f_mou_7         |            3.83863  | loc: Local calls  within same telecom circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 7: July,                             |
| loc_ic_mou_7             |            3.83863  | loc: Local calls  within same telecom circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                                                  |
| spl_og_mou_7             |            3.83863  | spl: Special calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                                                                            |
| std_ic_t2t_mou_7         |            3.83863  | std: STD calls  outside the calling circle, ic: Incoming calls, t2t: Operator T to T ie within same operator mobile to mobile, mou: Minutes of usage  voice calls, 7: July,     |
| isd_og_mou_7             |            3.83863  | isd: ISD calls, og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                                                                                |
| std_ic_t2m_mou_7         |            3.83863  | std: STD calls  outside the calling circle, ic: Incoming calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 7: July,                          |
| std_og_mou_7             |            3.83863  | std: STD calls  outside the calling circle, og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                                                    |
| std_ic_t2f_mou_7         |            3.83863  | std: STD calls  outside the calling circle, ic: Incoming calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 7: July,                               |
| std_og_t2m_mou_7         |            3.83863  | std: STD calls  outside the calling circle, og: Outgoing calls, t2m: Operator T to other operator mobile, mou: Minutes of usage  voice calls, 7: July,                          |
| std_ic_t2o_mou_7         |            3.83863  | std: STD calls  outside the calling circle, ic: Incoming calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls, 7: July,                      |
| std_og_t2c_mou_7         |            3.83863  | std: STD calls  outside the calling circle, og: Outgoing calls, t2c: Operator T to its own call center, mou: Minutes of usage  voice calls, 7: July,                            |
| std_ic_mou_7             |            3.83863  | std: STD calls  outside the calling circle, ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                                                    |
| std_og_t2f_mou_7         |            3.83863  | std: STD calls  outside the calling circle, og: Outgoing calls, t2f: Operator T to fixed lines of T, mou: Minutes of usage  voice calls, 7: July,                               |
| og_others_7              |            3.83863  | og: Outgoing calls, 7: July,                                                                                                                                                    |
| spl_ic_mou_7             |            3.83863  | spl: Special calls, ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                                                                            |
| date_of_last_rech_8      |            3.51576  | rech: Recharge, 8: August,                                                                                                                                                      |
| date_of_last_rech_7      |            1.76288  | rech: Recharge, 7: July,                                                                                                                                                        |
| date_of_last_rech_6      |            1.57288  | rech: Recharge, 6: June,                                                                                                                                                        |
| last_date_of_month_8     |            1.04716  | 8: August,                                                                                                                                                                      |
| loc_ic_t2o_mou           |            1.00287  | loc: Local calls  within same telecom circle, ic: Incoming calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls,                             |
| std_og_t2o_mou           |            1.00287  | std: STD calls  outside the calling circle, og: Outgoing calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls,                               |
| loc_og_t2o_mou           |            1.00287  | loc: Local calls  within same telecom circle, og: Outgoing calls, t2o: Operator T to other operator fixed line, mou: Minutes of usage  voice calls,                             |
| last_date_of_month_7     |            0.570008 | 7: July,                                                                                                                                                                        |
| vol_3g_mb_8              |            0        | vol: Mobile internet usage volume in MB, 3g: G network, 8: August,                                                                                                              |
| aon                      |            0        | Age on network  number of days the customer is using the operator T network                                                                                                     |
| vbc_3g_8                 |            0        | vbc: Volume based cost  when no specific scheme is not purchased and paid as per usage, 3g: G network, 8: August,                                                               |
| vbc_3g_7                 |            0        | vbc: Volume based cost  when no specific scheme is not purchased and paid as per usage, 3g: G network, 7: July,                                                                 |
| vbc_3g_6                 |            0        | vbc: Volume based cost  when no specific scheme is not purchased and paid as per usage, 3g: G network, 6: June,                                                                 |
| sachet_3g_8              |            0        | sachet: Service schemes with validity smaller than a month, 3g: G network, 8: August,                                                                                           |
| sachet_3g_7              |            0        | sachet: Service schemes with validity smaller than a month, 3g: G network, 7: July,                                                                                             |
| sachet_3g_6              |            0        | sachet: Service schemes with validity smaller than a month, 3g: G network, 6: June,                                                                                             |
| monthly_2g_6             |            0        | monthly: Service schemes with validity equivalent to a month, 2g: G network, 6: June,                                                                                           |
| monthly_2g_7             |            0        | monthly: Service schemes with validity equivalent to a month, 2g: G network, 7: July,                                                                                           |
| monthly_2g_8             |            0        | monthly: Service schemes with validity equivalent to a month, 2g: G network, 8: August,                                                                                         |
| sachet_2g_6              |            0        | sachet: Service schemes with validity smaller than a month, 2g: G network, 6: June,                                                                                             |
| sachet_2g_7              |            0        | sachet: Service schemes with validity smaller than a month, 2g: G network, 7: July,                                                                                             |
| sachet_2g_8              |            0        | sachet: Service schemes with validity smaller than a month, 2g: G network, 8: August,                                                                                           |
| monthly_3g_8             |            0        | monthly: Service schemes with validity equivalent to a month, 3g: G network, 8: August,                                                                                         |
| monthly_3g_7             |            0        | monthly: Service schemes with validity equivalent to a month, 3g: G network, 7: July,                                                                                           |
| monthly_3g_6             |            0        | monthly: Service schemes with validity equivalent to a month, 3g: G network, 6: June,                                                                                           |
| id                       |            0        |                                                                                                                                                                                 |
| vol_3g_mb_7              |            0        | vol: Mobile internet usage volume in MB, 3g: G network, 7: July,                                                                                                                |
| total_rech_num_7         |            0        | rech: Recharge, num: Number, 7: July,                                                                                                                                           |
| last_date_of_month_6     |            0        | 6: June,                                                                                                                                                                        |
| arpu_6                   |            0        | arpu: Average revenue per user, 6: June,                                                                                                                                        |
| arpu_7                   |            0        | arpu: Average revenue per user, 7: July,                                                                                                                                        |
| arpu_8                   |            0        | arpu: Average revenue per user, 8: August,                                                                                                                                      |
| total_og_mou_6           |            0        | og: Outgoing calls, mou: Minutes of usage  voice calls, 6: June,                                                                                                                |
| total_og_mou_7           |            0        | og: Outgoing calls, mou: Minutes of usage  voice calls, 7: July,                                                                                                                |
| total_og_mou_8           |            0        | og: Outgoing calls, mou: Minutes of usage  voice calls, 8: August,                                                                                                              |
| circle_id                |            0        | Telecom circle area to which the customer belongs to                                                                                                                            |
| total_ic_mou_6           |            0        | ic: Incoming calls, mou: Minutes of usage  voice calls, 6: June,                                                                                                                |
| total_ic_mou_7           |            0        | ic: Incoming calls, mou: Minutes of usage  voice calls, 7: July,                                                                                                                |
| total_ic_mou_8           |            0        | ic: Incoming calls, mou: Minutes of usage  voice calls, 8: August,                                                                                                              |
| total_rech_num_6         |            0        | rech: Recharge, num: Number, 6: June,                                                                                                                                           |
| total_rech_num_8         |            0        | rech: Recharge, num: Number, 8: August,                                                                                                                                         |
| vol_3g_mb_6              |            0        | vol: Mobile internet usage volume in MB, 3g: G network, 6: June,                                                                                                                |
| total_rech_amt_6         |            0        | rech: Recharge, amt: Amount in local currency, 6: June,                                                                                                                         |
| total_rech_amt_7         |            0        | rech: Recharge, amt: Amount in local currency, 7: July,                                                                                                                         |
| total_rech_amt_8         |            0        | rech: Recharge, amt: Amount in local currency, 8: August,                                                                                                                       |
| max_rech_amt_6           |            0        | max: Maximum, rech: Recharge, amt: Amount in local currency, 6: June,                                                                                                           |
| max_rech_amt_7           |            0        | max: Maximum, rech: Recharge, amt: Amount in local currency, 7: July,                                                                                                           |
| max_rech_amt_8           |            0        | max: Maximum, rech: Recharge, amt: Amount in local currency, 8: August,                                                                                                         |
| last_day_rch_amt_6       |            0        | amt: Amount in local currency, 6: June,                                                                                                                                         |
| last_day_rch_amt_7       |            0        | amt: Amount in local currency, 7: July,                                                                                                                                         |
| last_day_rch_amt_8       |            0        | amt: Amount in local currency, 8: August,                                                                                                                                       |
| vol_2g_mb_6              |            0        | vol: Mobile internet usage volume in MB, 2g: G network, 6: June,                                                                                                                |
| vol_2g_mb_7              |            0        | vol: Mobile internet usage volume in MB, 2g: G network, 7: July,                                                                                                                |
| vol_2g_mb_8              |            0        | vol: Mobile internet usage volume in MB, 2g: G network, 8: August,                                                                                                              |
| churn_probability        |            0        |                                                                                                                                                                                 |

In [15]:
# --- [CELL 14]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 15}
def GetHighMissingCols(df):
    high_missing = df.loc[(df["MissingPercentage"] > 70.0)]["index"]
    return high_missing

high_missing_cols = GetHighMissingCols(percent_missing)
telecom.loc[:,high_missing_cols].describe()

,date_of_last_rech_data_6,date_of_last_rech_data_7,date_of_last_rech_data_8,total_rech_data_6,total_rech_data_7,total_rech_data_8,max_rech_data_6,max_rech_data_7,max_rech_data_8,count_rech_2g_6,...,arpu_3g_8,arpu_2g_6,arpu_2g_7,arpu_2g_8,night_pck_user_6,night_pck_user_7,night_pck_user_8,fb_user_6,fb_user_7,fb_user_8
count,17568,17865,18417,17568.000000,17865.000000,18417.000000,17568.000000,17865.000000,18417.000000,17568.000000,...,18417.000000,17568.000000,17865.000000,18417.000000,17568.000000,17865.000000,18417.000000,17568.000000,17865.000000,18417.000000
mean,2014-06-19 01:06:08.852459264,2014-07-19 18:36:27.204030208,2014-08-19 12:07:46.786121728,2.467612,2.679989,2.652441,126.500000,126.402071,125.374925,1.865323,...,90.618564,86.863900,85.846074,86.348404,0.025273,0.024069,0.021013,0.916325,0.909544,0.890319
min,2014-06-01 00:00:00,2014-07-01 00:00:00,2014-08-01 00:00:00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,...,-24.490000,-35.830000,-13.090000,-55.830000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2014-06-12 00:00:00,2014-07-13 00:00:00,2014-08-12 00:00:00,1.000000,1.000000,1.000000,25.000000,25.000000,25.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
50%,2014-06-21 00:00:00,2014-07-22 00:00:00,2014-08-21 00:00:00,1.000000,2.000000,1.000000,145.000000,145.000000,145.000000,1.000000,...,0.840000,11.300000,8.800000,9.090000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
75%,2014-06-27 00:00:00,2014-07-28 00:00:00,2014-08-28 00:00:00,3.000000,3.000000,3.000000,177.000000,177.000000,179.000000,2.000000,...,122.070000,122.070000,122.070000,122.070000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
max,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-31 00:00:00,61.000000,54.000000,60.000000,1555.000000,1555.000000,1555.000000,42.000000,...,3716.900000,5054.350000,4809.360000,3483.170000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,NaN,NaN,NaN,2.794610,3.073472,3.101265,109.352573,109.459266,109.648799,2.566377,...,189.907986,171.321203,178.067280,170.297094,0.156958,0.153269,0.143432,0.276907,0.286842,0.312501


In [16]:
# --- [CELL 15]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 16}
recharge_cols_missing = ['total_rech_data_6','total_rech_data_7','total_rech_data_8',
                         'av_rech_amt_data_6','av_rech_amt_data_7','av_rech_amt_data_8']
def ReplaceRechargeColsForNoRecharge(df):
    for col in recharge_cols_missing:
        df[col] = df[col].replace(np.NaN,0.0)


ReplaceRechargeColsForNoRecharge(telecom)
ShowSummary(telecom)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69999 entries, 0 to 69998
Data columns (total 172 columns):
 #    Column                    Non-Null Count  Dtype         
---   ------                    --------------  -----         
 0    id                        69999 non-null  int64         
 1    circle_id                 69999 non-null  int64         
 2    loc_og_t2o_mou            69297 non-null  float64       
 3    std_og_t2o_mou            69297 non-null  float64       
 4    loc_ic_t2o_mou            69297 non-null  float64       
 5    last_date_of_month_6      69999 non-null  datetime64[ns]
 6    last_date_of_month_7      69600 non-null  datetime64[ns]
 7    last_date_of_month_8      69266 non-null  datetime64[ns]
 8    arpu_6                    69999 non-null  float64       
 9    arpu_7                    69999 non-null  float64       
 10   arpu_8                    69999 non-null  float64       
 11   onnet_mou_6               67231 non-null  float64       
 12   on

In [17]:
# --- [CELL 16]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 17}
# Calclate new column total recharge amount for data:  av_rech_amt_data * total_rech_data

#function that create total recharge amount data columns
def CreateTotalRechAmtDataCols(df):
    df['total_rech_amt_data_6'] = df.av_rech_amt_data_6 * df.total_rech_data_6
    df['total_rech_amt_data_7'] = df.av_rech_amt_data_7 * df.total_rech_data_7
    df['total_rech_amt_data_8'] = df.av_rech_amt_data_8 * df.total_rech_data_8

CreateTotalRechAmtDataCols(telecom)
telecom.describe()

,id,circle_id,loc_og_t2o_mou,std_og_t2o_mou,loc_ic_t2o_mou,last_date_of_month_6,last_date_of_month_7,last_date_of_month_8,arpu_6,arpu_7,...,fb_user_7,fb_user_8,aon,vbc_3g_8,vbc_3g_7,vbc_3g_6,churn_probability,total_rech_amt_data_6,total_rech_amt_data_7,total_rech_amt_data_8
count,69999.000000,69999.0,69297.0,69297.0,69297.0,69999,69600,69266,69999.000000,69999.000000,...,17865.000000,18417.000000,69999.000000,69999.000000,69999.000000,69999.00000,69999.000000,69999.000000,69999.000000,69999.000000
mean,34999.000000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00.000000256,2014-08-30 23:59:59.999999744,283.134365,278.185912,...,0.909544,0.890319,1220.639709,68.108597,65.935830,60.07674,0.101887,148.479217,172.524819,174.246063
min,0.000000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-31 00:00:00,-2258.709000,-1289.715000,...,0.000000,0.000000,180.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
25%,17499.500000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-31 00:00:00,93.581000,86.714000,...,1.000000,1.000000,468.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
50%,34999.000000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-31 00:00:00,197.484000,191.588000,...,1.000000,1.000000,868.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
75%,52498.500000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-31 00:00:00,370.791000,365.369500,...,1.000000,1.000000,1813.000000,0.000000,0.000000,0.00000,0.000000,8.000000,17.000000,23.000000
max,69998.000000,109.0,0.0,0.0,0.0,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-31 00:00:00,27731.088000,35145.834000,...,1.000000,1.000000,4337.000000,12916.220000,9165.600000,11166.21000,1.000000,55296.000000,55080.000000,89106.500000
std,20207.115084,0.0,0.0,0.0,0.0,NaN,NaN,NaN,334.213918,344.366927,...,0.286842,0.312501,952.426321,269.328659,267.899034,257.22681,0.302502,749.012768,856.608088,950.062467


In [18]:
# --- [CELL 17]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 18}
print(telecom.quantile(np.arange(0.9, 1.01, 0.01)))

            id  circle_id  loc_og_t2o_mou  std_og_t2o_mou  loc_ic_t2o_mou  \
0.90  62998.20      109.0             0.0             0.0             0.0   
0.91  63698.18      109.0             0.0             0.0             0.0   
0.92  64398.16      109.0             0.0             0.0             0.0   
0.93  65098.14      109.0             0.0             0.0             0.0   
0.94  65798.12      109.0             0.0             0.0             0.0   
0.95  66498.10      109.0             0.0             0.0             0.0   
0.96  67198.08      109.0             0.0             0.0             0.0   
0.97  67898.06      109.0             0.0             0.0             0.0   
0.98  68598.04      109.0             0.0             0.0             0.0   
0.99  69298.02      109.0             0.0             0.0             0.0   
1.00  69998.00      109.0             0.0             0.0             0.0   

     last_date_of_month_6 last_date_of_month_7 last_date_of_month_8  \
0.90

In [19]:
# --- [CELL 18]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 19}
#c
quantile_series = telecom.quantile(np.arange(0.9, 1.01, 0.01))
print(type(quantile_series))
print(quantile_series.index)
print(quantile_series.head())

<class 'pandas.core.frame.DataFrame'>
Index([               0.9,               0.91,               0.92,
                     0.93, 0.9400000000000001, 0.9500000000000001,
       0.9600000000000001, 0.9700000000000001, 0.9800000000000001,
       0.9900000000000001,                1.0],
      dtype='float64')
            id  circle_id  loc_og_t2o_mou  std_og_t2o_mou  loc_ic_t2o_mou  \
0.90  62998.20      109.0             0.0             0.0             0.0   
0.91  63698.18      109.0             0.0             0.0             0.0   
0.92  64398.16      109.0             0.0             0.0             0.0   
0.93  65098.14      109.0             0.0             0.0             0.0   
0.94  65798.12      109.0             0.0             0.0             0.0   

     last_date_of_month_6 last_date_of_month_7 last_date_of_month_8  \
0.90           2014-06-30           2014-07-31           2014-08-31   
0.91           2014-06-30           2014-07-31           2014-08-31   
0.92          

In [20]:
# --- [CELL 19]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 20}
#c
quantile_series.index = range(len(quantile_series))  # Replace with integer index

In [21]:
# --- [CELL 20]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 21}
#c
quantile_series.index = quantile_series.index.astype(str)

In [22]:
# --- [CELL 21]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 22}
# === BEFORE (original) ===
# #c
# percentage_change = quantile_series.pct_change().mul(100)

# === AFTER (edited) ===
numeric_quantile_series = quantile_series.select_dtypes(include=[np.number])
percentage_change = numeric_quantile_series.pct_change().mul(100)

In [23]:
import pandas as pd

assert "percentage_change" in globals(), "percentage_change was not created"

numeric_cols = quantile_series.select_dtypes(include="number").columns

# Must contain exactly numeric columns from quantile_series
assert list(percentage_change.columns) == list(numeric_cols), (
    "percentage_change should include only numeric columns from quantile_series"
)

# Must preserve row count/index and be numeric
assert percentage_change.index.equals(quantile_series.index), "index should be preserved"
assert all(pd.api.types.is_numeric_dtype(dt) for dt in percentage_change.dtypes), (
    "percentage_change should contain only numeric dtypes"
)

# Behavioral correctness of pct_change * 100
expected = quantile_series[numeric_cols].pct_change() * 100
pd.testing.assert_frame_equal(percentage_change, expected)